In [ ]:
import pandas as pd
import os

PROTOCOL_PATH = "LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"

df = pd.read_csv(
    PROTOCOL_PATH,
    sep=" ",
    header=None,
    names=["speaker", "utt_id", "system", "unused", "label"]
)

df["path"] = df["utt_id"].apply(
    lambda x: f"ASVspoof2019_LA/ASVspoof2019_LA_train/flac/{x}.flac"
)

df["label"] = df["label"].map({"bonafide": 0, "spoof": 1})

print(df.head())
print("Total samples:", len(df))
print("Fake:", df.label.sum(), "Real:", len(df)-df.label.sum())


In [ ]:
import pandas as pd
import os

ROOT = "LA"   # THIS is the real folder you have

PROTOCOL_PATH = os.path.join(
    ROOT,
    "ASVspoof2019_LA_cm_protocols",
    "ASVspoof2019.LA.cm.train.trn.txt"
)

df = pd.read_csv(
    PROTOCOL_PATH,
    sep=" ",
    header=None,
    names=["speaker", "utt_id", "system", "unused", "label"]
)

# auto-detect flac folder name (flac vs FLAC)
TRAIN_ROOT = os.path.join(ROOT, "ASVspoof2019_LA_train")
FLAC_DIR = "flac" if os.path.exists(os.path.join(TRAIN_ROOT, "flac")) else "FLAC"

df["path"] = df["utt_id"].apply(
    lambda x: os.path.join(TRAIN_ROOT, FLAC_DIR, f"{x}.flac")
)

df["label"] = df["label"].map({"bonafide": 0, "spoof": 1})

print(df.head())
print("Total samples:", len(df))
print("Fake:", df.label.sum(), "Real:", len(df) - df.label.sum())


In [ ]:
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset

SAMPLE_RATE = 16000
DURATION = 3
MAX_LEN = SAMPLE_RATE * DURATION

class ASVspoofDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, _ = librosa.load(row.path, sr=SAMPLE_RATE)

        # random crop
        if len(audio) > MAX_LEN:
            start = np.random.randint(0, len(audio) - MAX_LEN)
            audio = audio[start:start + MAX_LEN]
        else:
            audio = np.pad(audio, (0, MAX_LEN - len(audio)))

        attention_mask = np.ones(len(audio))

        return {
            "input_values": torch.tensor(audio, dtype=torch.float32),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": torch.tensor(row.label, dtype=torch.float32)
        }


In [ ]:
import torch.nn as nn
from transformers import WavLMModel

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")

        # FREEZE backbone first
        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(self.wavlm.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x, attention_mask):
        out = self.wavlm(x, attention_mask=attention_mask)
        pooled = out.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

dataset = ASVspoofDataset(df)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepfakeDetector().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5
)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
EPOCHS_FROZEN = 2

for epoch in range(EPOCHS_FROZEN):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        x = batch["input_values"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        logits = model(x, attention_mask=mask).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Frozen Epoch {epoch+1}/{EPOCHS_FROZEN} | Loss: {total_loss/len(loader):.4f}")

for param in model.wavlm.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Unfrozen WavLM — {trainable}/{total} parameters trainable")

optimizer = optim.AdamW(model.parameters(), lr=1e-5)

EPOCHS_UNFROZEN = 3 

for epoch in range(EPOCHS_UNFROZEN):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        x = batch["input_values"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        logits = model(x, attention_mask=mask).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Unfrozen Epoch {epoch+1}/{EPOCHS_UNFROZEN} | Loss: {total_loss/len(loader):.4f}")

torch.save(model.state_dict(), "wavlm_asvspoof_la.pth")
print("Model saved")

In [ ]:
import torch.nn as nn

class SmoothBCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # label smoothing
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepfakeDetector().to(device)

# LOAD ASVSPOOF TRAINED WEIGHTS
state = torch.load("wavlm_asvspoof_la.pth", map_location=device)
model.load_state_dict(state)

print("ASVspoof-trained model loaded")


In [ ]:
for param in model.wavlm.parameters():
    param.requires_grad = False

print("WavLM frozen (classifier will adapt to Hindi)")


In [ ]:
import pandas as pd
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

SAMPLE_RATE = 16000
DURATION = 3
MAX_LEN = SAMPLE_RATE * DURATION

HAV_ROOT = "hav-df/HAV-DF"
AUDIO_DIR = os.path.join(HAV_ROOT, "audio")

df = pd.read_csv(os.path.join(HAV_ROOT, "video_metadata.csv"))
df["label"] = df["label"].map({"REAL": 0, "FAKE": 1})
df["path"] = df["video_name"].apply(
    lambda x: os.path.join(AUDIO_DIR, x.replace(".mp4", ".wav"))
)

df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
print("HAV samples:", len(df))


In [ ]:
class HAVDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            audio, _ = librosa.load(row.path, sr=SAMPLE_RATE)
        except:
            return None

        if len(audio) > MAX_LEN:
            start = np.random.randint(0, len(audio) - MAX_LEN)
            audio = audio[start:start + MAX_LEN]
        else:
            audio = np.pad(audio, (0, MAX_LEN - len(audio)))

        return {
            "input_values": torch.tensor(audio, dtype=torch.float32),
            "label": torch.tensor(row.label, dtype=torch.float32)
        }

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch])
    }

loader = DataLoader(
    HAVDataset(df),
    batch_size=8,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)


In [ ]:
criterion = SmoothBCE(smoothing=0.1)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)


In [ ]:
from tqdm import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    steps = 0

    pbar = tqdm(loader, desc=f"HAV Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        if batch is None:
            continue

        x = batch["input_values"].to(device)
        y = batch["label"].to(device)

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1
        pbar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss/steps:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAMPLE_RATE = 16000
DURATION = 3
MAX_LEN = SAMPLE_RATE * DURATION
BATCH_SIZE = 8

HAV_ROOT = "hav-df/HAV-DF" 
META_PATH = os.path.join(HAV_ROOT, "video_metadata.csv")

hav_df = pd.read_csv(META_PATH)

hav_df["label"] = hav_df["label"].map({"REAL": 0, "FAKE": 1})

def build_path(name):
    train_path = os.path.join(HAV_ROOT, "train_videos", name)
    test_path  = os.path.join(HAV_ROOT, "test_videos", name)
    return train_path if os.path.exists(train_path) else test_path

hav_df["path"] = hav_df["video_name"].apply(build_path)

hav_df = hav_df[hav_df["path"].apply(os.path.exists)].reset_index(drop=True)

print(f"HAV samples: {len(hav_df)} | Real: {(hav_df.label==0).sum()} | Fake: {(hav_df.label==1).sum()}")

class HAVDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            audio, _ = librosa.load(row.path, sr=SAMPLE_RATE, mono=True)
        except:
            return None  # skip bad samples

        if len(audio) > MAX_LEN:
            start = np.random.randint(0, len(audio) - MAX_LEN)
            audio = audio[start:start+MAX_LEN]
        else:
            audio = np.pad(audio, (0, MAX_LEN - len(audio)))

        mask = np.ones(MAX_LEN, dtype=np.float32)

        return {
            "input_values": torch.tensor(audio, dtype=torch.float32),
            "attention_mask": torch.tensor(mask, dtype=torch.float32),
            "label": torch.tensor(row.label, dtype=torch.float32)
        }

# collate to drop None
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return {
        k: torch.stack([b[k] for b in batch])
        for k in batch[0]
    }

loader = DataLoader(
    HAVDataset(hav_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,     
    collate_fn=collate_fn,
    pin_memory=True
)

model = DeepfakeDetector().to(DEVICE)
model.load_state_dict(torch.load("wavlm_asvspoof_la.pth", map_location=DEVICE))


class SmoothBCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets)

criterion = SmoothBCE(0.1)

for p in model.wavlm.parameters():
    p.requires_grad = False

optimizer = optim.AdamW(model.classifier.parameters(), lr=3e-4)

print("Training classifier only")

for epoch in range(1):
    model.train()
    total_loss, steps = 0, 0

    for batch in tqdm(loader, desc="HAV"):
        if batch is None:
            continue

        x = batch["input_values"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        logits = model(x, attention_mask=mask).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f"Loss: {total_loss/steps:.4f}")

for name, p in model.wavlm.named_parameters():
    p.requires_grad = False
    if any(layer in name for layer in ["encoder.layers.8",
                                       "encoder.layers.9",
                                       "encoder.layers.10",
                                       "encoder.layers.11"]):
        p.requires_grad = True

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-6
)

print("Fine-tuning last WavLM layers")

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss, steps = 0, 0

    for batch in tqdm(loader, desc=f"HAV Epoch {epoch+1}/{EPOCHS}"):
        if batch is None:
            continue

        x = batch["input_values"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        logits = model(x, attention_mask=mask).squeeze(1)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss/steps:.4f}")


torch.save(model.state_dict(), "wavlm_asvspoof_hav_hindi.pth")
print("Model saved: wavlm_asvspoof_hav_hindi.pth")


In [ ]:
import torch
from transformers import WavLMModel
import torch.nn as nn
import librosa

class HAVModel(nn.Module):
    def __init__(self):
        super(HAVModel, self).__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")

        # Match training classifier exactly
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),  # 768 is WavLM hidden size
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)     # Single output (real/fake)
        )

    def forward(self, x):
        # x: waveform [batch, time]
        outputs = self.wavlm(x).last_hidden_state  # [batch, seq_len, hidden]
        features = outputs.mean(dim=1)            # mean pooling
        logits = self.classifier(features)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HAVModel()
model.load_state_dict(torch.load("wavlm_asvspoof_hav_hindi.pth", map_location=device))
model.to(device)
model.eval()

SAMPLE_RATE = 16000

def process_audio(audio_path):
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(device)
    return audio_tensor

def predict(audio_path):
    audio_tensor = process_audio(audio_path)
    with torch.no_grad():
        logits = model(audio_tensor)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
    label = "Real" if pred == 0 else "Fake"
    return label, probs.cpu().numpy()





In [ ]:
from pydub import AudioSegment
import os

def mp3_to_wav(mp3_path, wav_path):
    audio = AudioSegment.from_mp3(mp3_path)
    audio = audio.set_channels(1)  # mono
    audio.export(wav_path, format="wav")
    print(f"Converted {mp3_path} -> {wav_path}")

mp3_path = "/content/fake_004.mp3"
wav_path = "/content/fake_004.wav"
mp3_to_wav(mp3_path, wav_path)

In [ ]:
audio_file = "/content/fake_004.wav"
label, probs = predict(audio_file)
print(f"Prediction: {label}, Probabilities: {probs}")